<a href="https://colab.research.google.com/github/Mario-Cattaneo/SemesterProject/blob/main/Splitting.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [11]:
import os
import random
import torch
import torch.nn as nn
import torch.nn.functional as F
import torchvision.models as models
from torchvision.transforms import Resize, CenterCrop, ToTensor, Normalize, Compose
from PIL import Image

# 1. Clone the repository if it doesn't exist
REPO_DIR = "imagenet-sample-images"
REPO_URL = "https://github.com/EliSchwartz/imagenet-sample-images.git"

if not os.path.exists(REPO_DIR):
    print(f"Cloning {REPO_URL}...")
    os.system(f"git clone {REPO_URL} {REPO_DIR}")

# 2. Define the DAG Execution Engine

class DAGExecutionEngine:
    def __init__(self, model, branch_weights):
        self.model = model
        self.weights = branch_weights  # e.g., {'A': 0.6, 'B': 0.4}

    def execute_pipeline(self, x):
        """
        Executes the pipeline: P -> [A, B] -> S
        """
        # --- STEP 1: Predecessor Device 'P' ---
        # Device 'P' runs the first Conv + ReLU block on the full input
        x_P = self.model.features[0](x)  # Conv1_1
        x_P = self.model.features[1](x_P) # ReLU

        # --- STEP 2: Branching & Horizontal Splitting ---
        # The output of 'P' is split horizontally between 'A' and 'B'
        B, C, H, W = x_P.shape

        # Calculate split heights based on relative weights of A and B
        h_A = int(round(self.weights['A'] * H))
        h_B = H - h_A

        conv_layer = self.model.features[2] # Conv1_2 (kernel_size=3, padding=1)
        pad_size = conv_layer.kernel_size[0] // 2

        # Device 'A' (Top Branch) gets top slice + bottom overlap row from B
        slice_A = x_P[:, :, 0 : h_A + pad_size, :]
        # Device 'B' (Bottom Branch) gets bottom slice + top overlap row from A
        slice_B = x_P[:, :, h_A - pad_size : H, :]

        # --- STEP 3: Parallel Execution on 'A' and 'B' ---
        # Device 'A' manual padding: needs top padding, but gets bottom from overlap
        slice_A_padded = F.pad(slice_A, (1, 1, pad_size, 0))
        out_A = F.conv2d(slice_A_padded, conv_layer.weight, conv_layer.bias, stride=conv_layer.stride)
        out_A = self.model.features[3](out_A) # ReLU

        # Device 'B' manual padding: gets top from overlap, needs bottom padding
        slice_B_padded = F.pad(slice_B, (1, 1, 0, pad_size))
        out_B = F.conv2d(slice_B_padded, conv_layer.weight, conv_layer.bias, stride=conv_layer.stride)
        out_B = self.model.features[3](out_B) # ReLU

        # --- STEP 4: Successor Device 'S' (Merging & Rest of Network) ---
        # Device 'S' merges the outputs from 'A' and 'B'
        merged_features = torch.cat([out_A, out_B], dim=2)

        # Device 'S' runs the rest of the network sequentially
        x_rest = merged_features
        for layer in self.model.features[4:]:
            x_rest = layer(x_rest)

        x_rest = self.model.avgpool(x_rest)
        x_rest = torch.flatten(x_rest, 1)
        logits = self.model.classifier(x_rest)
        return logits

# 3. Self-Contained ImageNet Loader
def load_imagenet_samples(k=20):
    all_files = [f for f in os.listdir(REPO_DIR) if f.endswith(".JPEG")]
    unique_wnids = sorted(list(set(f.split("_")[0] for f in all_files)))
    wnid_to_idx = {wnid: idx for idx, wnid in enumerate(unique_wnids)}

    selected_files = random.sample(all_files, min(k, len(all_files)))

    transform = Compose([
        Resize(224),
        CenterCrop(224),
        ToTensor(),
        Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
    ])

    images, targets, names = [], [], []
    for filename in selected_files:
        parts = filename.replace(".JPEG", "").split("_")
        wnid = parts[0]
        class_name = " ".join(parts[1:])
        class_idx = wnid_to_idx[wnid]
        img_path = os.path.join(REPO_DIR, filename)
        try:
            img = Image.open(img_path).convert('RGB')
            images.append(transform(img))
            targets.append(class_idx)
            names.append(class_name)
        except Exception as e:
            print(f"Failed to load {filename}: {e}")

    return torch.stack(images), torch.tensor(targets), names

# 4. Main Evaluation Pipeline
def main():
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    print(f"Running evaluation on: {device}\n")

    # Load Pretrained VGG16
    print("Loading pretrained VGG16 model...")
    vgg16 = models.vgg16(weights=models.VGG16_Weights.DEFAULT).to(device).eval()

    # Define the DAG Topology Branch Weights (must sum to 1.0)
    # Output of 'P' is divided between 'A' and 'B' based on these weights
    branch_weights = {
        'A': 0.6,  # Device A handles 60% of the height
        'B': 0.4   # Device B handles 40% of the height
    }
    print(f"Configured DAG Topology: P -> [A ({branch_weights['A']}), B ({branch_weights['B']})] -> S")

    # Load k random ImageNet samples
    k_samples = 50
    print(f"\nLoading {k_samples} random ImageNet samples from local repository...")
    images, targets, names = load_imagenet_samples(k=k_samples)
    images, targets = images.to(device), targets.to(device)

    # Initialize the DAG Execution Engine
    engine = DAGExecutionEngine(vgg16, branch_weights)

    # Run Inference
    print("\nRunning inference...")
    with torch.no_grad():
        out_base = vgg16(images)  # Baseline: Unsplit sequential model
        out_dag = engine.execute_pipeline(images)  # DAG Pipeline execution

    # Verify mathematical equivalence of the DAG execution
    dag_matches = torch.allclose(out_base, out_dag, atol=1e-4)
    print(f"DAG Pipeline mathematically matches Baseline: {dag_matches}")

    # Calculate Accuracies
    correct_base = 0
    correct_dag = 0

    for i in range(len(names)):
        target_label = targets[i].item()
        pred_base = out_base[i].argmax().item()
        pred_dag = out_dag[i].argmax().item()

        if pred_base == target_label: correct_base += 1
        if pred_dag == target_label: correct_dag += 1

    # Print final accuracies
    total = len(names)
    print("\n" + "="*50)
    print(f"EVALUATION RESULTS ON {total} IMAGENET SAMPLES")
    print("="*50)
    print(f"Baseline Accuracy (Unsplit): {correct_base/total*100:.2f}%")
    print(f"DAG Pipeline Accuracy:       {correct_dag/total*100:.2f}%")
    print("="*50)

if __name__ == "__main__":
    main()

Running evaluation on: cpu

Loading pretrained VGG16 model...
Configured DAG Topology: P -> [A (0.6), B (0.4)] -> S

Loading 50 random ImageNet samples from local repository...

Running inference...
DAG Pipeline mathematically matches Baseline: True

EVALUATION RESULTS ON 50 IMAGENET SAMPLES
Baseline Accuracy (Unsplit): 86.00%
DAG Pipeline Accuracy:       86.00%
